# Neural Network Training for N-Body Prediction

This notebook implements five distinct deep learning architectures to model and predict the trajectories of an $N$-body gravitational system ($N=3$).

In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
import os
import time
import matplotlib.pyplot as plt
from google.colab import drive, files
from sklearn.model_selection import train_test_split

Ensure the path to your dataset location in Drive is correct.

In [ ]:
drive.mount('/content/drive', force_remount=True)
CSV_PATH = '/content/drive/MyDrive/Colab Notebooks/rk4_3body_20260604_1522.csv'

df = pd.read_csv(CSV_PATH)

all_ids = df['id_sim'].unique()

# Spliting into train/test the simulations IDs
train_ids, test_ids = train_test_split(all_ids, test_size=0.2, random_state=42)

train_ids_set = set(train_ids)
test_ids_set = set(test_ids)

print(f"Simulations for training: {len(train_ids)}")
print(f"Simulations for testing: {len(test_ids)}")

df_train = df[df['id_sim'].isin(train_ids)].copy()
df_test = df[df['id_sim'].isin(test_ids)].copy()

N = int((df.shape[1] - 2) / 5)
dt = float(df['time'].iloc[1] - df['time'].iloc[0])

##Presets

In [ ]:
MODELS = {
    1: "MLP",
    2: "LSTM",
    3: "GNN",
    4: "GNODE",
    5: "PI-GNN"
}

option = int(input("Select the model: 1 = MLP, 2 = LSTM, 3 = GNN, 4 = GNODE, 5 = PI-GNN\n"))
print("Selected model:", MODELS.get(option))

## Functions

### Custom Layers



To model non-linear celestial mechanics, options 3, 4, and 5 construct structured layers that override standard feedforward flows:

* **Edge Network:** Evaluates pairwise interactions $\phi_e(v_i, v_j)$ between relative states of a receiver body $i$ and a sender body $j$.
* **Message Masking & Aggregation:** Utilizes an identity mask matrix (`tf.eye`) to neutralize self-gravitation artifacts, subsequently reducing all neighbor dynamics via a summation pooling vector.
* **Node Network:** Updates individual body states based on aggregated neighborhood forces.
* **ODE Integration Layer:** Implements a localized integration routine to blend current velocities with predicted directional vectors over the fixed step size `dt`.
* **Physics Loss Constraints:** Calculates analytic classic Newtonian acceleration metrics alongside a mean squared data loss function to impose inductive physical biases.

In [ ]:
if option == 3:
  class GNN(tf.keras.layers.Layer):
    def __init__(self, dim, **kwargs):
      super().__init__(**kwargs)
      self.node = tf.keras.Sequential([
          tf.keras.layers.Dense(dim, activation='swish'),
          tf.keras.layers.Dense(4) # Output: x, y, vx, vy
      ])
      self.edge = tf.keras.Sequential([
          tf.keras.layers.Dense(dim, activation='swish'),
          tf.keras.layers.Dense(dim, activation='swish')
      ])

    def call(self, nodes):
      current_dtype = nodes.dtype
      N = tf.shape(nodes)[1]
      # nodes shape: [Batch, N, 5]

      # collect the data from body 1
      receiver_states = tf.expand_dims(nodes, axis=2) # [Batch, N, 1, 5]
      receiver_states = tf.repeat(receiver_states, N, axis=2) # [Batch, N, N, 5]

      # collect data from body 2
      sender_states = tf.expand_dims(nodes, axis=1) # [Batch, 1, N, 5]
      sender_states = tf.repeat(sender_states, N, axis=1) # [Batch, N, N, 5]

      # collect data from both bodies
      interaction_input = tf.concat([receiver_states, sender_states], axis=-1) # shape: [Batch, N, N, 10]
      # we put that data of the both bodies through the Edge Network
      pairwise_messages = self.edge(interaction_input) # [Batch, N, N, 256]

      # we create a 'mask' for interation of the type body 1 <-> body 1
      mask = tf.eye(N, dtype=current_dtype)
      mask = tf.reshape(mask, (1, N, N, 1))

      # apply the 'mask'
      pairwise_messages = pairwise_messages * (tf.cast(1.0, current_dtype) - mask) # [Batch, N, N, 256]
      # aggregate all incoming messages to single latent representation (256) for each body
      total_messages = tf.reduce_sum(pairwise_messages, axis=2) # [Batch, N, 256]

      # that latent representation is put through the node network
      return self.node(total_messages) # [Batch, N, 4]

In [ ]:
if option == 4:
  class Encoder(tf.keras.layers.Layer):
    def __init__(self, dim, **kwargs):
      super().__init__(**kwargs)
      self.dense = tf.keras.layers.Dense(dim, activation='relu')
      self.out = tf.keras.layers.Dense(dim) # Output layer has no activation
      self.norm = tf.keras.layers.LayerNormalization()

    def call(self, x):
      x = self.dense(x)
      x = self.out(x)
      return self.norm(x)

  class GNN(tf.keras.layers.Layer):
    def __init__(self, dim, **kwargs):
      super().__init__(**kwargs)
      self.node = tf.keras.Sequential([
          tf.keras.layers.Dense(dim, activation='swish'),
          tf.keras.layers.Dense(2) # Output: ax, ay
      ])
      self.edge = tf.keras.Sequential([
          tf.keras.layers.Dense(dim, activation='swish'),
          tf.keras.layers.Dense(dim, activation='swish')
      ])

    def call(self, nodes):
      current_dtype = nodes.dtype

      num_bodies = tf.shape(nodes)[1]

      receiver_states = tf.expand_dims(nodes, axis=2)
      receiver_states = tf.repeat(receiver_states, num_bodies, axis=2)

      sender_states = tf.expand_dims(nodes, axis=1)
      sender_states = tf.repeat(sender_states, num_bodies, axis=1)

      interaction_input = tf.concat([receiver_states, sender_states], axis=-1)
      pairwise_messages = self.edge(interaction_input)

      mask = tf.eye(num_bodies, dtype=current_dtype)
      mask = tf.reshape(mask, (1, num_bodies, num_bodies, 1))

      pairwise_messages = pairwise_messages * (tf.cast(1.0, current_dtype) - mask)
      total_messages = tf.reduce_sum(pairwise_messages, axis=2)

      return self.node(total_messages)

  class ODE(tf.keras.layers.Layer):
    def __init__(self, dt, **kwargs):
      super().__init__(**kwargs)
      self.dt = dt

    def call(self, current_state, accelerations):
      pos = current_state[:, :, 1:3]
      vel = current_state[:, :, 3:5]
      dt = self.dt

      predicted_pos = pos + vel * dt
      predicted_vel = vel + accelerations * dt

      avg_vel = 0.5 * (vel + predicted_vel)

      new_pos = pos + avg_vel * dt
      new_vel = vel + accelerations * dt

      return tf.concat([new_pos, new_vel], axis=-1)

In [ ]:
if option == 5:
  class GNN(tf.keras.layers.Layer):
    def __init__(self, dim, **kwargs):
      super().__init__(**kwargs)
      self.node = tf.keras.Sequential([
          tf.keras.layers.Dense(dim, activation='swish'),
          tf.keras.layers.Dense(2) # Output: ax, ay
      ])
      self.edge = tf.keras.Sequential([
          tf.keras.layers.Dense(dim, activation='swish'),
          tf.keras.layers.Dense(dim, activation='swish')
      ])

    def call(self, nodes):
      current_dtype = nodes.dtype

      N = tf.shape(nodes)[1]

      receiver_states = tf.expand_dims(nodes, axis=2)
      receiver_states = tf.repeat(receiver_states, N, axis=2)

      sender_states = tf.expand_dims(nodes, axis=1)
      sender_states = tf.repeat(sender_states, N, axis=1)

      interaction_input = tf.concat([receiver_states, sender_states], axis=-1)
      pairwise_messages = self.edge(interaction_input)

      mask = tf.eye(N, dtype=current_dtype)
      mask = tf.reshape(mask, (1, N, N, 1))

      pairwise_messages = pairwise_messages * (tf.cast(1.0, current_dtype) - mask)
      total_messages = tf.reduce_sum(pairwise_messages, axis=2)

      return self.node(total_messages)

  def compute_acceleration(inputs, G=1.0):
    N = tf.shape(inputs)[1]

    states = tf.reshape(inputs, (-1, N, 5))
    masses = states[:, :, 0:1]
    positions = states[:, :, 1:3]

    pos_i = tf.tile(tf.expand_dims(positions, axis=2), [1, 1, N, 1])
    pos_j = tf.tile(tf.expand_dims(positions, axis=1), [1, N, 1, 1])
    r_ij = pos_j - pos_i

    dist_sq = tf.reduce_sum(tf.square(r_ij), axis=-1, keepdims=True) + 0.05 # epsilon = 0.05
    dist = tf.sqrt(dist_sq)

    # Evaluate classic acceleration tensors
    mass_j = tf.tile(tf.expand_dims(masses, axis=1), [1, N, 1, 1])
    acc_components = G * mass_j * r_ij / (dist_sq * dist)

    # Mask self-gravitation elements
    mask = 1.0 - tf.eye(N, dtype=tf.float32)
    mask = tf.expand_dims(tf.expand_dims(mask, axis=0), axis=-1)
    acc_components = acc_components * mask

    acc = tf.reduce_sum(acc_components, axis=2) # Shape: (Batch, N, 2)
    return acc

  optimizer = tf.keras.optimizers.Adam(learning_rate=5e-4)

  @tf.function
  def train_step(X_batch, Y_batch, dt=0.001, lambda_=0.1):
    N = tf.shape(X_batch)[1]
    states_t = tf.reshape(X_batch, (-1, N, 5))
    pos_t = states_t[:, :, 1:3]
    vel_t = states_t[:, :, 3:5]

    states_next = tf.reshape(Y_batch, (-1, N, 4))

    with tf.GradientTape() as tape:
      # Forward Pass: Predict acceleration vectors [ax, ay]
      pred_acc = model(X_batch)

      # Kinematic integration projection
      pred_vel_next = vel_t + (pred_acc * dt)
      pred_pos_next = pos_t + (vel_t * dt)

      # Combine predictions to match your Y_batch shape layout (Batch, N, 4)
      predicted_state_next = tf.concat([pred_pos_next, pred_vel_next], axis=-1)

      # Calculate losses
      loss_data = tf.reduce_mean(tf.square(predicted_state_next - states_next))
      physics_acc = compute_acceleration(X_batch, G=1.0)
      loss_physics = tf.reduce_mean(tf.square(pred_acc - physics_acc))

      total_loss = loss_data + (lambda_ * loss_physics)

    gradients = tape.gradient(total_loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    # Return total loss alongside the state prediction for metrics tracking
    return total_loss, predicted_state_next

  @tf.function
  def val_step(X_batch, Y_batch, dt=0.001, lambda_=0.1):
    N = tf.shape(X_batch)[1]
    states_t = tf.reshape(X_batch, (-1, N, 5))
    pos_t = states_t[:, :, 1:3]
    vel_t = states_t[:, :, 3:5]

    states_next = tf.reshape(Y_batch, (-1, N, 4))

    # Forward Pass without tracking gradients (inference mode)
    pred_acc = model(X_batch)

    pred_vel_next = vel_t + (pred_acc * dt)
    pred_pos_next = pos_t + (vel_t * dt)
    predicted_state_next = tf.concat([pred_pos_next, pred_vel_next], axis=-1)

    # Track the exact validation losses using identical constraints
    loss_data = tf.reduce_mean(tf.square(predicted_state_next - states_next))
    physics_acc = compute_acceleration(X_batch, G=1.0)
    loss_physics = tf.reduce_mean(tf.square(pred_acc - physics_acc))

    val_total_loss = loss_data + (lambda_ * loss_physics)
    return val_total_loss, predicted_state_next

### Data Preparation


Data processing alters depending on the sequential requirements of the model topologies:

1. **Standard Relational Split (Options 1, 3, 4, 5):** Parses independent frame pairs $(t, t+1)$. Features for training input ($X$) preserve all 5 dimensions $[m, x, y, v_x, v_y]$, while targets ($Y$) isolate the resulting 4 positional and velocity variables.
2. **Temporal Window Split (Option 2):** Reshapes consecutive row histories into 3D tensors of shape `(Batch, SEQ_LENGTH, Features)` to supply the recurrent gates of the LSTM network.

In [ ]:
def prepare_data(df_train, df_test, option, N=3, seq_length=10):
  def split_standard(dataframe):
    X_list, Y_list = [], []
    for _, group in dataframe.groupby('id_sim'):
      data = group.sort_values('time').iloc[:, 2:].values
      if len(data) < 2: continue

      y_mask = [i for i in range(data.shape[1]) if i % 5 != 0]
      X_list.append(data[:-1])
      Y_list.append(data[1:, y_mask])

    return np.vstack(X_list), np.vstack(Y_list)

  def split_sequence(dataframe, seq_len):
    X_list, Y_list = [], []
    num_features = dataframe.shape[1] - 2
    y_mask = [j for j in range(num_features) if j % 5 != 0]

    for _, group in dataframe.groupby('id_sim'):
      data = group.sort_values('time').iloc[:, 2:].values
      if len(data) <= seq_len: continue

      for i in range(len(data) - seq_len):
        X_list.append(data[i : i + seq_len])
        Y_list.append(data[i + seq_len, y_mask])

    return np.array(X_list), np.array(Y_list)

  if option == 2:
    X_train, Y_train = split_sequence(df_train, seq_length)
    X_test, Y_test = split_sequence(df_test, seq_length)
  elif option in [1, 3, 4, 5]:
    X_train, Y_train = split_standard(df_train)
    X_test, Y_test = split_standard(df_test)
  else: raise ValueError("Invalid option! Choose between 1 and 5.")

  if option in [3, 4, 5]:
    X_train = X_train.reshape(-1, N, 5).astype('float32')
    Y_train = Y_train.reshape(-1, N, 4).astype('float32')
    X_test = X_test.reshape(-1, N, 5).astype('float32')
    Y_test = Y_test.reshape(-1, N, 4).astype('float32')

  print(f"X_train dimensions: {X_train.shape} | Y_train dimensions: {Y_train.shape}")
  print(f"X_test dimensions:  {X_test.shape} | Y_test dimensions:  {Y_test.shape}\n")

  return X_train, Y_train, X_test, Y_test

###Training Model

The `define_model` module abstracts architecture creation.
* Options 1–4 automatically compile via standard Keras optimizers using Mean Squared Error (**MSE**) loss profiles.
* Option 5 initializes a native functional graph layout ready to accept customized low-level backpropagation passes inside a custom `GradientTape` session.

In [ ]:
def define_model(option, X_train, Y_train, seq_length):
  if option in [1, 2, 3, 4]:
    if option == 1:
      model = tf.keras.Sequential([
          # Input
          tf.keras.layers.Input(shape=(X_train.shape[1],)),
          # Hidden layers
          tf.keras.layers.Dense(256, activation='swish'),
          tf.keras.layers.Dense(256, activation='swish'),
          tf.keras.layers.Dense(256, activation='swish'),
          tf.keras.layers.Dense(256, activation='swish'),
          tf.keras.layers.Dense(128, activation='swish'),
          # Output
          tf.keras.layers.Dense(Y_train.shape[1])
      ])

    elif option == 2:
      model = tf.keras.Sequential([
          tf.keras.layers.Input(shape=(seq_length, X_train.shape[2])),
          # First layer returns sequences for the next LSTM
          tf.keras.layers.LSTM(256, return_sequences=True),
          # Second layer returns what it learned from the sequence
          tf.keras.layers.LSTM(128, return_sequences=False),
          tf.keras.layers.Dense(128, activation='swish'),
          tf.keras.layers.Dense(Y_train.shape[1])
      ])

    elif option == 3:
      inputs = tf.keras.Input(shape=(None, 5))
      outputs = GNN(dim=256)(inputs)
      model = tf.keras.Model(inputs=inputs, outputs=outputs)

    elif option == 4:
      inputs = tf.keras.Input(shape=(None, 5))
      encoder = Encoder(dim=128)
      gnn = GNN(dim=128)
      ode = ODE(dt=dt)
      latent_values = encoder(inputs)
      accelerations = gnn(latent_values)
      outputs = ode(inputs, accelerations)
      model = tf.keras.Model(inputs=inputs, outputs=outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss='mse',
        metrics=['mae']
    )

  elif option == 5:
    inputs = tf.keras.Input(shape=(None, 5))
    outputs = GNN(dim=128)(inputs)
    model = tf.keras.Model(inputs=inputs, outputs=outputs)

  model.summary()
  return model

The training engine processes iterations through two disparate pipelines:
* **High-Level Fit Loop (Options 1-4):** Employs standard production-ready compiled `model.fit()` protocols with integrated validation performance tracking.
* **Custom Step Loop (Option 5):** Manages a dedicated operational loop feeding batches via `tf.data.Dataset`. It extracts spatial positions, applies custom regularization constraints ($\lambda_{physics}$), dynamically logs tracking metrics, and explicitly steps the optimizer.

In [ ]:
def train_model(model, option, X_train, Y_train, X_test, Y_test,
                epochs=100, batch_size=512, train_step_fn=None, val_step_fn=None):

  if option in [1, 2, 3, 4]:
    print(f"Training model {MODELS.get(option)} for {epochs} epochs...")
    history_obj = model.fit(
      X_train, Y_train,
      epochs=epochs,
      batch_size=batch_size,
      validation_data=(X_test, Y_test),
      verbose=1
    )
    return history_obj.history

  elif option == 5:
    print(f"Training model {MODELS.get(option)} for {epochs} epochs...")
    if train_step_fn is None or val_step_fn is None:
      raise ValueError("You must pass 'train_step_fn' and 'val_step_fn' when running PI-GNN.")


    train_dataset = tf.data.Dataset.from_tensor_slices((X_train, Y_train))
    train_dataset = train_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    test_dataset = tf.data.Dataset.from_tensor_slices((X_test, Y_test))
    test_dataset = test_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    train_loss_metric = tf.keras.metrics.Mean(name='train_loss')
    train_mae_metric = tf.keras.metrics.MeanAbsoluteError(name='train_mae')

    val_loss_metric = tf.keras.metrics.Mean(name='val_loss')
    val_mae_metric = tf.keras.metrics.MeanAbsoluteError(name='val_mae')

    history = {
        'loss': [], 'mae': [],
        'val_loss': [], 'val_mae': []
    }


    for epoch in range(epochs):
      start_time = time.time()

      train_loss_metric.reset_state()
      train_mae_metric.reset_state()
      val_loss_metric.reset_state()
      val_mae_metric.reset_state()

      # TRAINING PHASE
      for step_X, step_Y in train_dataset:
        loss_value, predicted_states = train_step_fn(step_X, step_Y)
        train_loss_metric.update_state(loss_value)
        train_mae_metric.update_state(step_Y, predicted_states)

      # VALIDATION PHASE
      for step_X_val, step_Y_val in test_dataset:
        v_loss, val_preds = val_step_fn(step_X_val, step_Y_val)
        val_loss_metric.update_state(v_loss)
        val_mae_metric.update_state(step_Y_val, val_preds)

      epoch_time = time.time() - start_time

      epoch_loss = train_loss_metric.result().numpy()
      epoch_mae = train_mae_metric.result().numpy()
      epoch_val_loss = val_loss_metric.result().numpy()
      epoch_val_mae = val_mae_metric.result().numpy()

      history['loss'].append(epoch_loss)
      history['mae'].append(epoch_mae)
      history['val_loss'].append(epoch_val_loss)
      history['val_mae'].append(epoch_val_mae)

      print(f"Epoch {epoch+1:02d}/{EPOCHS} [{epoch_time:.1f}s] -> "
            f"loss: {epoch_loss:.6f} | mae: {epoch_mae:.6f} | "
            f"val_loss: {epoch_val_loss:.6f} | val_mae: {epoch_val_mae:.6f}")

    return history

  else: raise ValueError("Invalid option selected. Choose between 1 and 5.")

Evaluating model performance purely on isolated frame transformations can mask error-compounding issues. The simulation engine performs a realistic test:
1. **Autoregressive Feedback:** Extracts an initial true coordinate step from the test split and projects forward for hundreds of steps (`steps=300`). At every step, the network's output is processed, appended with invariants (mass), and recycled as the next input frame.
2. **Visual Verification:** Plots the ground truth numerical integration path against the predicted AI trajectories across all $N$ objects over a long-horizon coordinate plane.

In [ ]:
def evaluate_model(model, option, history, X_test, Y_test):

  mae_key = 'mae' if 'mae' in history else 'mean_absolute_error'
  val_loss_key = 'val_loss' if 'val_loss' in history else 'val_mean_absolute_error'
  val_mae_key = 'val_mae' if 'val_mae' in history else 'val_mean_absolute_error'

  train_loss = history['loss'][-1]
  train_mae = history[mae_key][-1]

  test_loss = history[val_loss_key][-1]
  test_mae = history[val_mae_key][-1]

  print(f"Evaluation metrics for model {MODELS.get(option)} training:")
  print(f"{'Metric':<15} {'Training':<15} {'Test':<15}")
  print(f"{'Loss (MSE)':<15} {train_loss:<15.10f} {test_loss:<15.10f}")
  print(f"{'MAE':<15} {train_mae:<15.10f} {test_mae:<15.10f}")

###Simulation Example

In [ ]:
def run_simulation(model, df_test, option, chosen_id, steps, N=3, dt=0.001, seq_length=10):

  sim_data = df_test[df_test['id_sim'] == chosen_id].sort_values('time')

  if sim_data.empty: raise ValueError(f"Simulation ID {chosen_id} not found in the test DataFrame.")

  if option == 1:
    initial_state = sim_data.iloc[0, 2:].values.reshape(1, -1)
    mass_indices = [k * 5 for k in range(N)]
    masses = initial_state[0, mass_indices]
    temp_input = initial_state

  elif option == 2:
    initial_window = sim_data.iloc[0:seq_length, 2:].values
    mass_indices = [k * 5 for k in range(N)]
    masses = initial_window[0, mass_indices]
    temp_input = np.expand_dims(initial_window, axis=0)

  elif option in [3, 4, 5]:
    initial_state = sim_data.iloc[0, 2:].values.reshape(1, N, 5).astype('float32')
    temp_input = np.copy(initial_state)

  ai_path = []
  print(f"Running simulation for {MODELS.get(option)} ({steps} steps)...")

  for _ in range(steps):

    if option in [1, 2, 3, 4]:
      prediction = model.predict(temp_input, verbose=0)
      ai_path.append(prediction[0]) # Shape: (N, 4) or flat

      if option == 1:
        next_input = np.zeros((1, N * 5))
        for i in range(N):
          next_input[0, i*5] = masses[i]
          next_input[0, i*5+1 : 5*(i+1)] = prediction[0, i*4 : 4*(i+1)]
        temp_input = next_input

      elif option == 2:
        next_frame = np.zeros((1, N * 5))
        for i in range(N):
            next_frame[0, i*5] = masses[i]
            next_frame[0, i*5+1 : 5*(i+1)] = prediction[0, i*4 : 4*(i+1)]
        temp_input = np.append(temp_input[:, 1:, :], np.expand_dims(next_frame, axis=0), axis=1)

      elif option in [3, 4]:
        temp_input = np.concatenate([temp_input[:, :, 0:1], prediction], axis=-1)

    elif option == 5:
      masses_pi = temp_input[:, :, 0:1]
      pos_t     = temp_input[:, :, 1:3]
      vel_t     = temp_input[:, :, 3:5]

      pred_acc = model(tf.convert_to_tensor(temp_input), training=False).numpy()

      vel_next = vel_t + (pred_acc * dt)
      pos_next = pos_t + (vel_t * dt)

      step_decoded = np.concatenate([pos_next, vel_next], axis=-1)[0]
      ai_path.append(step_decoded)

      temp_input = np.concatenate([masses_pi, pos_next, vel_next], axis=-1)

  ai_path = np.array(ai_path)

  if option in [1, 2]:
    ai_path = ai_path.reshape(-1, N, 4)

  x_real, y_real = [], []
  x_ai, y_ai = [], []
  sim_values = sim_data.iloc[:, 2:].values.reshape(-1, N, 5)

  start_offset = seq_length if option == 2 else 1

  for i in range(N):
    x_real.append(sim_values[start_offset : start_offset + steps, i, 1])
    y_real.append(sim_values[start_offset : start_offset + steps, i, 2])
    x_ai.append(ai_path[:, i, 0])
    y_ai.append(ai_path[:, i, 1])

  return x_real, y_real, x_ai, y_ai

In [ ]:
def plotting(x_real, y_real, x_ai, y_ai, N, option):
  fig, ax = plt.subplots(figsize=(8, 8))
  ax.set_aspect('equal')

  all_x = np.concatenate([np.array(x).flatten() for x in x_ai + x_real])
  all_y = np.concatenate([np.array(y).flatten() for y in y_ai + y_real])

  x_min, x_max = np.min(all_x), np.max(all_x)
  y_min, y_max = np.min(all_y), np.max(all_y)

  margin_x = (x_max - x_min) * 0.15 if x_max != x_min else 1.0
  margin_y = (y_max - y_min) * 0.15 if y_max != y_min else 1.0

  ax.set_xlim(x_min - margin_x, x_max + margin_x)
  ax.set_ylim(y_min - margin_y, y_max + margin_y)

  cmap = plt.get_cmap('tab10')

  for i in range(N):
    color = cmap(i % 10)

    ax.plot(x_real[i], y_real[i], color=color, linestyle='--', alpha=0.4, label=f'Body {i+1} (Numerical)')
    ax.plot(x_ai[i], y_ai[i], color=color, linestyle='-', linewidth=2, label=f'Body {i+1} ({MODELS.get(option)})')

    ax.scatter(x_real[i][-1], y_real[i][-1], facecolors='none', edgecolors=color, s=100, zorder=5)
    ax.scatter(x_ai[i][-1], y_ai[i][-1], color=color, s=100, zorder=5)

  ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
  ax.set_title(f"Numerical VS. {MODELS.get(option)} N-Body System Prediction")
  ax.set_xlabel("x, AU")
  ax.set_ylabel("y, AU")
  ax.grid(True, linestyle=':', alpha=0.5)

  plt.tight_layout()
  plt.show()

##Compilation

In [ ]:
EPOCHS = 100
BATCH_SIZE = 512

SEQ_LENGTH = 10 # LSTM

X_train, Y_train, X_test, Y_test = prepare_data(df_train, df_test, option, N, SEQ_LENGTH)
model = define_model(option, X_train, Y_train, SEQ_LENGTH)
history = train_model(model=model, option=option,
    X_train=X_train, Y_train=Y_train,
    X_test=X_test, Y_test=Y_test,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    train_step_fn=lambda x, y: train_step(x, y, dt=dt, lambda_=0.1),
    val_step_fn=lambda x, y: val_step(x, y, dt=dt, lambda_=0.1)
)

evaluate_model(model, option, history, X_test, Y_test)

In [ ]:
x_real, y_real, x_ai, y_ai = run_simulation(model, df_test, option, 0, 1000, N, dt)
plotting(x_real, y_real, x_ai, y_ai, N, option)

In [ ]:
save_model = input("Save the model? [y/n]: ")

if save_model.strip().lower() in ['y', 'yes']:
  print("Connecting to Google Drive...")
  drive.mount('/content/drive', force_remount=True)

  filename = f'model_{MODELS.get(option)}_nbody.keras'

  drive_path = os.path.join('/content/drive/MyDrive/Colab Notebooks', filename)
  model.save(drive_path)
  print(f"Successfully saved to Google Drive at: {drive_path}")

  local_path = os.path.join('/content', filename)
  model.save(local_path)
  print(f"Local copy generated. Triggering browser download...")

  files.download(local_path)

elif save_model.strip().lower() in ['n', 'no']:
    print("Model saving skipped.")
else:
    print("Invalid input. Proceeding without saving the model.")